In [26]:
import tensorflow as tf
from tensorflow.keras import layers, models


In [27]:
import zipfile, os

zip_path = "/content/tiny-imagenet.zip"
extract_dir = "/content/tiny-imagenet"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

train_dir = os.path.join(extract_dir, 'train')
val_dir = os.path.join(extract_dir, 'val')
test_dir = os.path.join(extract_dir, 'test')

In [28]:
import tensorflow as tf

IMG_WIDTH = 64
IMG_HEIGHT = 64
BATCH_SIZE = 64
def load_image_dataset(directory, subset_name, shuffle=True):
    dataset = tf.keras.utils.image_dataset_from_directory(
        directory,image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,shuffle=shuffle,seed=42)
    return dataset


train_ds = load_image_dataset(train_dir, 'train')
val_ds = load_image_dataset(val_dir, 'validation', shuffle=False)
test_ds = load_image_dataset(test_dir, 'test', shuffle=False)
for images, labels in train_ds.take(1):
    print(f"Shape of images in a batch: {images.shape}")


Found 3500 files belonging to 10 classes.
Found 500 files belonging to 10 classes.
Found 1000 files belonging to 10 classes.
Shape of images in a batch: (64, 64, 64, 3)


In [29]:
from tensorflow.keras.models import Model


In [30]:
input_shape = train_ds.element_spec[0].shape[1:]
print(f"Input shape for the model: {input_shape}")

Input shape for the model: (64, 64, 3)


MODEL WITH PADDING

In [ ]:
inputs = tf.keras.Input(input_shape, name="Input")

x = layers.Conv2D(16, (3,3), padding='same', activation='relu', name="Conv1")(inputs)
print("After Conv1:", x.shape)

x = layers.MaxPooling2D((2,2), name="Pool1")(x)
print("After Pool1:", x.shape)

x = layers.Conv2D(32, (3,3), padding='same', activation='relu', name="Conv2")(x)
print("After Conv2:", x.shape)

x = layers.MaxPooling2D((2,2), name="Pool2")(x)
print("After Pool2:", x.shape)

x = layers.Conv2D(64, (3,3), padding='same', activation='relu', name="Conv3")(x)
print("After Conv3:", x.shape)

x = layers.Flatten(name="Flatten")(x)
print("After Flatten:", x.shape)

outputs = layers.Dense(10, activation='softmax', name="FC")(x)
print("After FC:", outputs.shape)

model_padding = Model(inputs, outputs, name="with_padding")


After Conv1: (None, 64, 64, 16)
After Pool1: (None, 32, 32, 16)
After Conv2: (None, 32, 32, 32)
After Pool2: (None, 16, 16, 32)
After Conv3: (None, 16, 16, 64)
After Flatten: (None, 16384)
After FC: (None, 10)


In [32]:
model.summary()


Model: "GeometryCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv1 (Conv2D)                  │ (None, 64, 64, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2 (Conv2D)                  │ (None, 64, 64, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv3 (Conv2D)                  │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 262144)         │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,584 (92.12 KB)

 Trainable params: 23,584 (92.12 KB)

 Non-trainable params: 0 (0.00 B)

MODEL WITHOUT PADDING

In [ ]:
inputs = tf.keras.Input(input_shape, name="Input")

x = layers.Conv2D(16, (3,3), padding='valid', activation='relu', name="Conv1")(inputs)
print("After Conv1:", x.shape)

x = layers.MaxPooling2D((2,2), name="Pool1")(x)
print("After Pool1:", x.shape)

x = layers.Conv2D(32, (3,3), padding='valid', activation='relu', name="Conv2")(x)
print("After Conv2:", x.shape)

x = layers.MaxPooling2D((2,2), name="Pool2")(x)
print("After Pool2:", x.shape)

x = layers.Conv2D(64, (3,3), padding='valid', activation='relu', name="Conv3")(x)
print("After Conv3:", x.shape)

x = layers.Flatten(name="Flatten")(x)
print("After Flatten:", x.shape)

outputs = layers.Dense(10, activation='softmax', name="FC")(x)
print("After FC:", outputs.shape)

model_no_padding = Model(inputs, outputs, name="without_padding")


After Conv1: (None, 62, 62, 16)
After Pool1: (None, 31, 31, 16)
After Conv2: (None, 29, 29, 32)
After Pool2: (None, 14, 14, 32)
After Conv3: (None, 12, 12, 64)
After Flatten: (None, 9216)
After FC: (None, 10)


In [34]:
model.summary()


Model: "GeometryCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv1 (Conv2D)                  │ (None, 64, 64, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2 (Conv2D)                  │ (None, 64, 64, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv3 (Conv2D)                  │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 262144)         │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,584 (92.12 KB)

 Trainable params: 23,584 (92.12 KB)

 Non-trainable params: 0 (0.00 B)

MODEL WITHOUT POOLING AND WITHOUT PADDING


In [ ]:
inputs = tf.keras.Input(input_shape, name="Input")

x = layers.Conv2D(16, (3,3), padding='valid', activation='relu', name="Conv1")(inputs)
print("After Conv1:", x.shape)

x = layers.Conv2D(32, (3,3), padding='valid', activation='relu', name="Conv2")(x)
print("After Conv2:", x.shape)

x = layers.Conv2D(64, (3,3), padding='valid', activation='relu', name="Conv3")(x)
print("After Conv3:", x.shape)

x = layers.Flatten(name="Flatten")(x)
print("After Flatten:", x.shape)

outputs = layers.Dense(10, activation='softmax', name="FC")(x)
print("After FC:", outputs.shape)

model_no_pool_no_pad = Model(inputs, outputs, name="no_pool_no_padding")


After Conv1: (None, 62, 62, 16)
After Conv2: (None, 60, 60, 32)
After Conv3: (None, 58, 58, 64)
After Flatten: (None, 215296)
After FC: (None, 10)


In [36]:
model.summary()

Model: "GeometryCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv1 (Conv2D)                  │ (None, 64, 64, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2 (Conv2D)                  │ (None, 64, 64, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv3 (Conv2D)                  │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 262144)         │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,584 (92.12 KB)

 Trainable params: 23,584 (92.12 KB)

 Non-trainable params: 0 (0.00 B)

MODEL WITHOUT POOLING AND WITH PADDING

In [ ]:
inputs = tf.keras.Input(input_shape, name="Input")

x = layers.Conv2D(16, (3,3), padding='same', activation='relu', name="Conv1")(inputs)
print("After Conv1:", x.shape)

x = layers.Conv2D(32, (3,3), padding='same', activation='relu', name="Conv2")(x)
print("After Conv2:", x.shape)

x = layers.Conv2D(64, (3,3), padding='same', activation='relu', name="Conv3")(x)
print("After Conv3:", x.shape)

x = layers.Flatten(name="Flatten")(x)
print("After Flatten:", x.shape)

outputs = layers.Dense(10, activation='softmax', name="FC")(x)
print("After FC:", outputs.shape)

model_no_pool_pad = Model(inputs, outputs, name="no_pool_padding")


After Conv1: (None, 64, 64, 16)
After Conv2: (None, 64, 64, 32)
After Conv3: (None, 64, 64, 64)
After Flatten: (None, 262144)
After FC: (None, 10)


In [38]:
model.summary()

Model: "GeometryCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv1 (Conv2D)                  │ (None, 64, 64, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2 (Conv2D)                  │ (None, 64, 64, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv3 (Conv2D)                  │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 262144)         │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,584 (92.12 KB)

 Trainable params: 23,584 (92.12 KB)

 Non-trainable params: 0 (0.00 B)

In [39]:
import pandas as pd

def model_stats(model):
    trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
    non_trainable = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
    total = trainable + non_trainable
    return total, trainable, non_trainable


FINAL COMPARISON


In [40]:
models_dict = {
    "With Padding": model_padding,
    "Without Padding": model_no_padding,
    "No Pool + No Padding": model_no_pool_no_pad,
    "No Pool + Padding": model_no_pool_pad
}

rows = []

for name, m in models_dict.items():
    total, trainable, non_trainable = model_stats(m)
    rows.append([name, total, trainable, non_trainable])

df = pd.DataFrame(rows, columns=["Model", "Total Params", "Trainable", "Non-trainable"])
df


,Model,Total Params,Trainable,Non-trainable
0,With Padding,187434,187434,0
1,Without Padding,115754,115754,0
2,No Pool + No Padding,2176554,2176554,0
3,No Pool + Padding,2645034,2645034,0
